In [1]:
!git clone https://github.com/mit-han-lab/torchquantum.git

Cloning into 'torchquantum'...
remote: Enumerating objects: 16339, done.
remote: Counting objects: 100% (2023/2023), done.
remote: Compressing objects: 100% (568/568), done.
remote: Total 16339 (delta 1706), reused 1486 (delta 1453), pack-reused 14316 (from 5)
Receiving objects: 100% (16339/16339), 106.59 MiB | 19.57 MiB/s, done.
Resolving deltas: 100% (9241/9241), done.


In [2]:
%cd torchquantum

/content/torchquantum


In [3]:
!pip install --editable .

Obtaining file:///content/torchquantum
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
INFO: pip is looking at multiple versions of pathos to determine which version is compatible with other requirements. This could take a while.
INFO: pip is looking at multiple versions of multiprocess to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.6/9.6 MB 73.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.2/14.2 MB 119.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 98.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 541.5/541.5 kB 27.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 79.2 MB/s  0:00:00
  Building editable for torchquantum (pyproject.toml) ... done
  Created wheel for t

In [4]:
%cd /content/torchquantum/examples/mnist/

/content/torchquantum/examples/mnist


If you are using Google Colab to run this notebook, upload the QNN.py and train_compare.py code in the path /content/torchquantum/examples/mnist/.

In [5]:
!python QNN.py

python3: can't open file '/content/torchquantum/examples/mnist/QNN.py': [Errno 2] No such file or directory


In [6]:
import os
# Check the available example directories
examples_path = '/content/torchquantum/examples'
if os.path.exists(examples_path):
    print("Examples found:", os.listdir(examples_path))
else:
    print("Examples directory not found at", examples_path)

Examples found: ['grover', 'mnist', 'encoder_examples', 'qiskit', 'clifford_qnn', 'save_load_example', 'optimal_control', 'index.rst', 'hadamard_grad', 'superdense_coding', 'qubit_rotation', 'train_state_prep', 'train_unitary_prep', 'converter_tq_qiskit', 'quantumnat', 'regression', 'quanvolution', '__init__.py', 'qaoa', 'param_shift_onchip_training', 'README.md', 'density', 'vqe', 'quantum_lstm', 'amplitude_encoding_mnist', 'PauliSumOp']


In [7]:
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
import numpy as np
from torchquantum.dataset import CIFAR10
import torchquantum as tq
import torchquantum.functional as tqf

# 1. Load CIFAR10 Dataset from torchquantum
dataset = CIFAR10(
    root='./data',
    train_valid_split_ratio=[0.9, 0.1],
    n_use_pairs=True,  # binary classification (typically classes 0 and 1)
    center_crop=24,
    resize=4  # Resize to 4x4 (16 pixels) to fit in quantum qubits
)

train_loader = torch.utils.data.DataLoader(dataset['train'], batch_size=4, shuffle=True)
test_loader = torch.utils.data.DataLoader(dataset['test'], batch_size=4, shuffle=False)

# 2. Plot samples from CIFAR10
dataiter = iter(train_loader)
images, labels = next(dataiter)

fig, axes = plt.subplots(1, 4, figsize=(10, 3))
for idx, (img, label) in enumerate(zip(images, labels)):
    # CIFAR-10 is RGB (3 channels), let's reshape and plot
    # Note: torchquantum's CIFAR10 returns downsampled images
    img_np = img.numpy()
    # Normalize to 0-1 for plotting if necessary
    img_min, img_max = img_np.min(), img_np.max()
    if img_max > img_min:
        img_np = (img_np - img_min) / (img_max - img_min)

    axes[idx].imshow(np.transpose(img_np, (1, 2, 0)))
    axes[idx].set_title(f"Label: {label.item()}")
    axes[idx].axis('off')
plt.tight_layout()
plt.show()

# 3. Create a Simple Quantum Neural Network (QNN) Model
class CIFARQNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.n_wires = 4
        self.q_device = tq.QuantumDevice(n_wires=self.n_wires)
        self.encoder = tq.GeneralEncoder(
            [
                {"input_idx": [0], "func": "rx", "wires": [0]},
                {"input_idx": [1], "func": "rx", "wires": [1]},
                {"input_idx": [2], "func": "rx", "wires": [2]},
                {"input_idx": [3], "func": "rx", "wires": [3]},
            ]
        )
        self.q_layer = tq.RandomLayer(n_ops=8, wires=[0, 1, 2, 3])
        self.measure = tq.MeasureAll(tq.PauliZ)

    def forward(self, x):
        # x shape: [batch_size, 3, 4, 4]
        # Reduce dimensions / flatten or use a mean channel to fit encoder inputs
        x = x.mean(dim=1).view(x.shape[0], -1) # Mean channel, [batch_size, 16]
        # Let's take the first 4 elements for simplicity of wire encoding
        x_input = x[:, :4]

        self.q_device.reset_states(x.shape[0])
        self.encoder(self.q_device, x_input)
        self.q_layer(self.q_device)
        out = self.measure(self.q_device)
        return out[:, :2]  # output 2 classes

model = CIFARQNN()
optimizer = optim.Adam(model.parameters(), lr=0.1)
criterion = nn.CrossEntropyLoss()

# 4. Train the QNN model on a small subset
model.train()
for epoch in range(3):
    total_loss = 0
    correct = 0
    total = 0
    for batch_idx, (data, target) in enumerate(train_loader):
        if batch_idx > 20: # Train on a small subset for quick output demonstration
            break
        optimizer.zero_grad()
        output = model(data)
        loss = criterion(output, target)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        _, predicted = output.max(1)
        total += target.size(0)
        correct += predicted.eq(target).sum().item()

    print(f"Epoch {epoch+1}, Loss: {total_loss/(batch_idx+1):.4f}, Accuracy: {correct/total*100:.2f}%")

ModuleNotFoundError: No module named 'torchquantum.dataset'

In [ ]:
import torch
import torchvision
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
import numpy as np

# Define the transformations for CIFAR-10
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

# Load the training set
trainset = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=transform)
trainloader = torch.utils.data.DataLoader(trainset, batch_size=8, shuffle=True)

# CIFAR-10 class labels
classes = ('plane', 'car', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck')

# Function to show an image
def imshow(img):
    img = img / 2 + 0.5  # unnormalize
    npimg = img.numpy()
    plt.imshow(np.transpose(npimg, (1, 2, 0)))
    plt.axis('off')

# Get some random training images
dataiter = iter(trainloader)
images, labels = next(dataiter)

# Show images and labels
fig = plt.figure(figsize=(12, 6))
imshow(torchvision.utils.make_grid(images))
plt.title(" ".join(f'{classes[labels[j]]:5s}' for j in range(8)))
plt.show()

In [ ]:
# Create the data directory
!mkdir -p ./data

# Download CIFAR-10 from a reliable academic mirror to speed up the process
!wget -O ./data/cifar-10-python.tar.gz https://datasets.activeloop.ai/cifar-10-python.tar.gz


In [ ]:
# Run this to extract and verify dataset loading without slow downloads
import torch
import torchvision
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
import numpy as np

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

# This will now detect the already downloaded file in ./data and extract it immediately
trainset = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=transform)
trainloader = torch.utils.data.DataLoader(trainset, batch_size=8, shuffle=True)

classes = ('plane', 'car', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck')

def imshow(img):
    img = img / 2 + 0.5
    npimg = img.numpy()
    plt.imshow(np.transpose(npimg, (1, 2, 0)))
    plt.axis('off')

dataiter = iter(trainloader)
images, labels = next(dataiter)

fig = plt.figure(figsize=(12, 6))
imshow(torchvision.utils.make_grid(images))
plt.title(" ".join(f'{classes[labels[j]]:5s}' for j in range(8)))
plt.show()

In [8]:
import sys

# Writing train_compare.py WITHOUT Exponential Moving Average (EMA) smoothing
code_content = """
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
import torchquantum as tq
from torchquantum.device.devices import QuantumDevice

# Set device
device = torch.device(\"cuda\" if torch.cuda.is_available() else \"cpu\")

# 1. Trainable Quantum Classifier Design
class QuantumClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.n_wires = 4
        self.q_device = QuantumDevice(n_wires=self.n_wires)
        self.encoder = tq.GeneralEncoder([
            {\"input_idx\": [0], \"func\": \"rx\", \"wires\": [0]},
            {\"input_idx\": [1], \"func\": \"rx\", \"wires\": [1]},
            {\"input_idx\": [2], \"func\": \"rx\", \"wires\": [2]},
            {\"input_idx\": [3], \"func\": \"rx\", \"wires\": [3]},
        ])

        self.rot_params0 = nn.Parameter(torch.zeros(4, 3))
        self.rot_params1 = nn.Parameter(torch.zeros(4, 3))

        self.rx = tq.RX()
        self.ry = tq.RY()
        self.rz = tq.RZ()
        self.cnot = tq.CNOT()
        self.measure = tq.MeasureAll(tq.PauliZ)

    def forward(self, x):
        if x.shape[1] == 3:
            x = x.mean(dim=1, keepdim=True)
        x = x.view(x.shape[0], -1)
        self.q_device.reset_states(x.shape[0])
        self.encoder(self.q_device, x)

        # Layer 0
        for i in range(self.n_wires):
            self.rx(self.q_device, wires=i, params=self.rot_params0[i, 0].unsqueeze(0))
            self.ry(self.q_device, wires=i, params=self.rot_params0[i, 1].unsqueeze(0))
            self.rz(self.q_device, wires=i, params=self.rot_params0[i, 2].unsqueeze(0))
        for i in range(self.n_wires):
            self.cnot(self.q_device, wires=[i, (i + 1) % self.n_wires])

        # Layer 1
        for i in range(self.n_wires):
            self.rx(self.q_device, wires=i, params=self.rot_params1[i, 0].unsqueeze(0))
            self.ry(self.q_device, wires=i, params=self.rot_params1[i, 1].unsqueeze(0))
            self.rz(self.q_device, wires=i, params=self.rot_params1[i, 2].unsqueeze(0))
        for i in range(self.n_wires):
            self.cnot(self.q_device, wires=[i, (i + 1) % self.n_wires])

        out = self.measure(self.q_device)
        return out[:, :2]

# 2. Data Preparation
transform = transforms.Compose([
    transforms.Resize((2, 2)),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

# MNIST Dataloader
mnist_train = torchvision.datasets.MNIST(root='./data', train=True, download=True, transform=transform)
mnist_test = torchvision.datasets.MNIST(root='./data', train=False, download=True, transform=transform)
mnist_train_idx = (mnist_train.targets == 0) | (mnist_train.targets == 1)
mnist_test_idx = (mnist_test.targets == 0) | (mnist_test.targets == 1)
mnist_train.data, mnist_train.targets = mnist_train.data[mnist_train_idx], mnist_train.targets[mnist_train_idx]
mnist_test.data, mnist_test.targets = mnist_test.data[mnist_test_idx], mnist_test.targets[mnist_test_idx]

mnist_train_loader = torch.utils.data.DataLoader(mnist_train, batch_size=128, shuffle=True)
mnist_test_loader = torch.utils.data.DataLoader(mnist_test, batch_size=128, shuffle=False)

# CIFAR-10 Dataloader
cifar_train = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=transform)
cifar_test = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=transform)
cifar_train_targets = torch.tensor(cifar_train.targets)
cifar_test_targets = torch.tensor(cifar_test.targets)
cifar_train_idx = (cifar_train_targets == 0) | (cifar_train_targets == 1)
cifar_test_idx = (cifar_test_targets == 0) | (cifar_test_targets == 1)
cifar_train.data = cifar_train.data[cifar_train_idx.numpy()]
cifar_train.targets = cifar_train_targets[cifar_train_idx].tolist()
cifar_test.data = cifar_test.data[cifar_test_idx.numpy()]
cifar_test.targets = cifar_test_targets[cifar_test_idx].tolist()

cifar_train_loader = torch.utils.data.DataLoader(cifar_train, batch_size=128, shuffle=True)
cifar_test_loader = torch.utils.data.DataLoader(cifar_test, batch_size=128, shuffle=False)

# Helper function to train and evaluate
def train_and_eval(train_loader, test_loader, epochs=50):
    model = QuantumClassifier().to(device)
    optimizer = optim.Adam(model.parameters(), lr=0.002, weight_decay=1e-3)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    criterion = nn.CrossEntropyLoss()
    loss_history = []
    accuracy_history = []

    for epoch in range(epochs):
        model.train()
        running_loss = 0.0
        count = 0
        for data, target in train_loader:
            data, target = data.to(device), target.to(device)
            optimizer.zero_grad()
            output = model(data)
            loss = criterion(output, target)
            loss.backward()
            optimizer.step()
            running_loss += loss.item()
            count += 1

        scheduler.step()
        loss_history.append(running_loss / count)

        # Validation
        model.eval()
        correct = 0
        total = 0
        with torch.no_grad():
            for data, target in test_loader:
                data, target = data.to(device), target.to(device)
                output = model(data)
                _, predicted = output.max(1)
                total += target.size(0)
                correct += predicted.eq(target).sum().item()

        acc = (correct / total) * 100
        accuracy_history.append(acc)
        if (epoch + 1) % 5 == 0 or epoch == 0:
            print(f"Epoch {epoch+1:02d}/{epochs}: Loss = {loss_history[-1]:.4f}, Accuracy = {acc:.2f}%")

    return loss_history, accuracy_history

# 3. Running Trainings
epochs = 50
print(\"=== Training QNN on MNIST (Binary) ===\")
mnist_losses, mnist_accs = train_and_eval(mnist_train_loader, mnist_test_loader, epochs=epochs)

print(\"\\n=== Training QNN on CIFAR-10 (Binary) ===\")
cifar_losses, cifar_accs = train_and_eval(cifar_train_loader, cifar_test_loader, epochs=epochs)

# 4. Plot four graphs (MNIST loss/accuracy and CIFAR loss/accuracy)
fig, axs = plt.subplots(2, 2, figsize=(14, 10))

# Graph 1: MNIST Accuracy (Raw Only)
axs[0, 0].plot(range(1, epochs + 1), mnist_accs, color='blue', linewidth=2, label='MNIST Val Acc')
axs[0, 0].set_title('MNIST Validation Accuracy')
axs[0, 0].set_xlabel('Epochs')
axs[0, 0].set_ylabel('Accuracy (%)')
axs[0, 0].grid(True)
axs[0, 0].legend()

# Graph 2: MNIST Loss
axs[0, 1].plot(range(1, epochs + 1), mnist_losses, label='MNIST Train Loss', color='cyan', linewidth=2)
axs[0, 1].set_title('MNIST Training Loss')
axs[0, 1].set_xlabel('Epochs')
axs[0, 1].set_ylabel('Loss')
axs[0, 1].grid(True)
axs[0, 1].legend()

# Graph 3: CIFAR-10 Accuracy (Raw Only)
axs[1, 0].plot(range(1, epochs + 1), cifar_accs, color='red', linewidth=2, label='CIFAR-10 Val Acc')
axs[1, 0].set_title('CIFAR-10 Validation Accuracy')
axs[1, 0].set_xlabel('Epochs')
axs[1, 0].set_ylabel('Accuracy (%)')
axs[1, 0].grid(True)
axs[1, 0].legend()

# Graph 4: CIFAR-10 Loss
axs[1, 1].plot(range(1, epochs + 1), cifar_losses, label='CIFAR-10 Train Loss', color='orange', linewidth=2)
axs[1, 1].set_title('CIFAR-10 Training Loss')
axs[1, 1].set_xlabel('Epochs')
axs[1, 1].set_ylabel('Loss')
axs[1, 1].grid(True)
axs[1, 1].legend()

plt.tight_layout()
plt.savefig('comparison_curve.png')
plt.close()
print("Four comparison graphs generated and saved to comparison_curve.png")
"""

with open('train_compare.py', 'w') as f:
    f.write(code_content)

print("Script 'train_compare.py' updated with raw accuracy curves (smoothing removed)!")


Script 'train_compare.py' updated with raw accuracy curves (smoothing removed)!


In [ ]:
# Execute the comparative script with raw accuracy lines only (no EMA smoothing)
!PYTHONPATH=/content/torchquantum python train_compare.py

from PIL import Image
import matplotlib.pyplot as plt
import os
if os.path.exists('comparison_curve.png'):
    img = Image.open('comparison_curve.png')
    plt.figure(figsize=(12, 9))
    plt.imshow(img)
    plt.axis('off')
    plt.show()
else:
    print("Error: Plot image not found.")

/content/torchquantum/torchquantum/functional/gate_wrapper.py:220: SyntaxWarning: invalid escape sequence '\d'
  Compute U \rho U^\dagger
/content/torchquantum/torchquantum/functional/gate_wrapper.py:299: SyntaxWarning: invalid escape sequence '\d'
  Compute \rho U^\dagger
100% 9.91M/9.91M [00:00<00:00, 127MB/s]
100% 28.9k/28.9k [00:00<00:00, 14.7MB/s]
100% 1.65M/1.65M [00:00<00:00, 121MB/s]
100% 4.54k/4.54k [00:00<00:00, 17.2MB/s]
 16% 27.7M/170M [03:11<17:44, 134kB/s]